In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQ_E.xpt"
csv_file = DATA_DIR / "SMQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 SMQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMQ020",
    "SMD030",
    "SMQ040",
    "SMQ050Q",
    "SMQ050U",
    "SMD055",
    "SMD057",
    "SMQ077",
    "SMD641",
    "SMD650",
    "SMD093",
    "SMDUPCA",
    "SMD100BR",
    "SMD100FL",
    "SMD100MN",
    "SMD100LN",
    "SMD100TR",
    "SMD100NI",
    "SMD100CO",
    "SMQ620",
    "SMD630",
    "SMQ660",
    "SMQ664M",
    "SMQ664C",
    "SMQ664W",
    "SMQ664B",
    "SMQ664O",
    "SMQ666M",
    "SMQ666C",
    "SMQ666K",
    "SMQ666W",
    "SMQ666B",
    "SMQ666S",
    "SMQ666O",
    "SMQ670",
    "SMAQUEX2"
]


if df.shape != (7145, 37):
    raise ValueError(
        f"Unexpected SMQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "SMQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: SMQ_E structure = "
    "7,145 rows x 37 columns."
)


# ============================================================
# 4. FIELD CLASSIFICATION
# ============================================================

#
# TEXT:
#
# SMDUPCA  = cigarette UPC
# SMD100BR = cigarette brand/sub-brand
#
# MEASUREMENTS:
#
# SMD100TR = FTC tar content
# SMD100NI = FTC nicotine content
# SMD100CO = FTC carbon monoxide content
#
# The measurement fields stay source-decoded FLOATS,
# even when an observed value is mathematically whole.
#
# Example:
#
# SMD100TR = 12.0 -> "12.0"
# SMD100NI = 1.0  -> "1.0"
# SMD100CO = 13.0 -> "13.0"
#
# All remaining numeric variables are questionnaire
# identifiers/codes/counts and can use clean integer style
# after source validation.
#

TEXT_COLUMNS = [
    "SMDUPCA",
    "SMD100BR"
]


MEASUREMENT_COLUMNS = [
    "SMD100TR",
    "SMD100NI",
    "SMD100CO"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if (
        col not in TEXT_COLUMNS
        and col not in MEASUREMENT_COLUMNS
    )
]


print(
    "\nInteger/code/count fields:",
    len(INTEGER_COLUMNS)
)

print(
    "Measurement fields preserved AS IS:",
    MEASUREMENT_COLUMNS
)

print(
    "Text fields preserved AS IS:",
    TEXT_COLUMNS
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY-ZERO VALUE
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


NUMERIC_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in TEXT_COLUMNS
]


tiny_counts = (
    df[NUMERIC_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[NUMERIC_COLUMNS]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "True decoded numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY PATTERN
# ============================================================

EXPECTED_TINY_COUNTS = {
    "SMD030": 122,
    "SMD641": 123,
    "SMD100FL": 7,
    "SMD100MN": 882
}


EXPECTED_TINY_TOTAL = 1134


actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != EXPECTED_TINY_TOTAL:
    raise ValueError(
        "Unexpected total tiny-artifact count."
    )


if true_zero_total != 0:
    raise ValueError(
        "Unexpected true zeros already decoded "
        "in original XPT."
    )


print(
    "\nPASS: Exact tiny-value pattern confirmed."
)


# ============================================================
# 7. CDC ZERO-SAFETY CHECK
# ============================================================

#
# CDC confirms:
#
# SMD030:
# 0 = Never smoked cigarettes regularly
#
# SMD641:
# Valid range 0-30 days
#
# SMD100FL:
# 0 = Non-filter
#
# SMD100MN:
# 0 = Non-menthol
#
# ONLY these four tiny-affected variables are approved.
#

APPROVED_ZERO_COLUMNS = {
    "SMD030",
    "SMD641",
    "SMD100FL",
    "SMD100MN"
}


if set(
    actual_tiny_counts
) != APPROVED_ZERO_COLUMNS:

    raise ValueError(
        "STOP: Tiny values found in "
        "unexpected SMQ_E variables."
    )


print(
    "PASS: All tiny values occur in "
    "CDC zero-permitted variables."
)


# ============================================================
# 8. RESTORE VALIDATED ZEROS
# ============================================================

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, expected_count in (
    EXPECTED_TINY_COUNTS.items()
):

    mask = df[col].eq(
        TINY_VALUE
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:
        raise ValueError(
            f"{col}: expected {expected_count:,} "
            f"tiny values, found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{actual_count:,} tiny -> 0"
    )


print(
    "PASS: All 1,134 validated zeros restored."
)


# ============================================================
# 9. FINAL ZERO COUNTS
# ============================================================

EXPECTED_ZERO_COUNTS = {
    "SMD030": 122,
    "SMD641": 123,
    "SMD100FL": 7,
    "SMD100MN": 882
}


for col, expected_count in (
    EXPECTED_ZERO_COUNTS.items()
):

    actual_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:
        raise ValueError(
            f"{col}: zero count differs.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


final_zero_total = int(
    df[NUMERIC_COLUMNS]
    .eq(0)
    .sum()
    .sum()
)


if final_zero_total != 1134:
    raise ValueError(
        "Unexpected zeros introduced outside "
        "the approved four fields."
    )


print(
    "PASS: Final zero counts exact."
)

print(
    "PASS: No unexpected zeros introduced."
)


# ============================================================
# 10. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df[NUMERIC_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:
    raise ValueError(
        "Tiny XPORT artifact remains."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:
    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:
    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):
    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 12. FRACTIONAL-VALUE AUDIT
# ============================================================

fractional_columns = {}


for col in NUMERIC_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:
        fractional_columns[
            col
        ] = fractional_count


EXPECTED_FRACTIONAL_COUNTS = {
    "SMD100NI": 872
}


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    fractional_columns
)


if fractional_columns != EXPECTED_FRACTIONAL_COUNTS:
    raise ValueError(
        "Unexpected fractional-value pattern.\n"
        f"Expected: {EXPECTED_FRACTIONAL_COUNTS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: SMD100NI is the only "
    "genuinely fractional field."
)

print(
    "PASS: SMD100NI fractional values = 872."
)


# ============================================================
# 13. VERIFY INTEGER/CODE FIELDS ARE WHOLE
# ============================================================

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=0
        )
    ):
        raise ValueError(
            f"STOP: {col} contains "
            "unexpected non-whole values."
        )


print(
    "PASS: All integer/code/count fields "
    "contain whole-number values."
)


# ============================================================
# 14. CDC CHECK - SMD030
# ============================================================

#
# CDC:
#
# 7-74 = 2,667
# 0    = 122
# 80   = 0
# 777  = 0
# 999  = 13
# Missing = 4,343
#

s = df["SMD030"]


if int(
    s.between(
        7,
        74
    ).sum()
) != 2667:

    raise ValueError(
        "SMD030 range count differs."
    )


if int(
    s.eq(0).sum()
) != 122:

    raise ValueError(
        "SMD030 zero count differs."
    )


if int(
    s.eq(80).sum()
) != 0:

    raise ValueError(
        "Unexpected SMD030=80."
    )


if int(
    s.eq(777).sum()
) != 0:

    raise ValueError(
        "Unexpected SMD030=777."
    )


if int(
    s.eq(999).sum()
) != 13:

    raise ValueError(
        "SMD030=999 count differs."
    )


if int(
    s.isna().sum()
) != 4343:

    raise ValueError(
        "SMD030 missing count differs."
    )


print(
    "\nPASS: SMD030 matches CDC."
)


# ============================================================
# 15. CDC CHECK - SMD641
# ============================================================

#
# CDC:
#
# 0-30 = 1,596
# 77   = 0
# 99   = 2
# Missing = 5,547
#
# Actual source zero count = 123
#

s = df["SMD641"]


if int(
    s.between(
        0,
        30
    ).sum()
) != 1596:

    raise ValueError(
        "SMD641 0-30 count differs."
    )


if int(
    s.eq(0).sum()
) != 123:

    raise ValueError(
        "SMD641 zero count differs."
    )


if int(
    s.eq(77).sum()
) != 0:

    raise ValueError(
        "Unexpected SMD641=77."
    )


if int(
    s.eq(99).sum()
) != 2:

    raise ValueError(
        "SMD641=99 count differs."
    )


if int(
    s.isna().sum()
) != 5547:

    raise ValueError(
        "SMD641 missing count differs."
    )


print(
    "PASS: SMD641 matches CDC."
)


# ============================================================
# 16. CDC CHECK - FILTER / MENTHOL
# ============================================================

EXPECTED_SMD100FL = {
    0: 7,
    1: 1249
}


EXPECTED_SMD100MN = {
    0: 882,
    1: 374
}


for col, expected in [
    (
        "SMD100FL",
        EXPECTED_SMD100FL
    ),
    (
        "SMD100MN",
        EXPECTED_SMD100MN
    )
]:

    actual = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual = {
        int(k): int(v)
        for k, v in actual.items()
    }


    if actual != expected:
        raise ValueError(
            f"{col} frequency mismatch.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    if int(
        df[col]
        .isna()
        .sum()
    ) != 5889:

        raise ValueError(
            f"{col} missing count differs."
        )


print(
    "PASS: SMD100FL matches CDC."
)

print(
    "PASS: SMD100MN matches CDC."
)


# ============================================================
# 17. MEASUREMENT FIELD CHECKS
# ============================================================

#
# These are cigarette-content measurements.
#
# They stay FLOAT-style even when mathematically whole.
#

MEASUREMENT_EXPECTATIONS = {

    "SMD100TR": {
        "valid": 934,
        "missing": 6211,
        "min": 1.0,
        "max": 25.0
    },

    "SMD100NI": {
        "valid": 934,
        "missing": 6211,
        "min": 0.1,
        "max": 1.8
    },

    "SMD100CO": {
        "valid": 934,
        "missing": 6211,
        "min": 1.0,
        "max": 19.0
    }
}


for col, cfg in (
    MEASUREMENT_EXPECTATIONS.items()
):

    s = df[col]


    if int(
        s.notna()
        .sum()
    ) != cfg["valid"]:

        raise ValueError(
            f"{col} valid count differs."
        )


    if int(
        s.isna()
        .sum()
    ) != cfg["missing"]:

        raise ValueError(
            f"{col} missing count differs."
        )


    if not np.isclose(
        s.min(),
        cfg["min"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col} minimum differs."
        )


    if not np.isclose(
        s.max(),
        cfg["max"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{col} maximum differs."
        )


    print(
        f"PASS: {col} measurement range/count."
    )


# ============================================================
# 18. LEGITIMATE MEASUREMENT ".0" COUNTS
# ============================================================

#
# Actual attached XPT:
#
# SMD100TR:
# every valid result is mathematically whole,
# but it is a measurement -> preserve x.0
#
# SMD100NI:
# 872 fractional + 62 whole-number float results
#
# SMD100CO:
# every valid result is mathematically whole,
# but it is a measurement -> preserve x.0
#

EXPECTED_MEASUREMENT_DOT_ZERO = {
    "SMD100TR": 934,
    "SMD100NI": 62,
    "SMD100CO": 934
}


print(
    "\n--- LEGITIMATE MEASUREMENT .0 CHECK ---"
)


for col, expected_count in (
    EXPECTED_MEASUREMENT_DOT_ZERO.items()
):

    actual_count = sum(
        str(
            float(value)
        ).endswith(".0")
        for value in df[col].dropna()
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:
        raise ValueError(
            f"{col}: measurement .0 count differs.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: Legitimate measurement .0 "
    "patterns confirmed."
)


# ============================================================
# 19. TEXT FIDELITY - SMDUPCA
# ============================================================

#
# IMPORTANT:
#
# SMDUPCA MUST REMAIN STRING/TEXT.
#
# A large number of UPCs begin with zero.
#
# Example:
#
# 028200007841
#
# Converting to numeric would destroy the leading zero.
#

upc = (
    df["SMDUPCA"]
    .astype(str)
)


upc_nonblank = (
    upc[
        upc != ""
    ]
)


if len(
    upc_nonblank
) != 817:

    raise ValueError(
        "SMDUPCA populated count differs."
    )


# Every populated source token is length 12.
if not (
    upc_nonblank
    .str.len()
    .eq(12)
    .all()
):

    raise ValueError(
        "Unexpected SMDUPCA token length."
    )


upc_no_match_count = int(
    upc_nonblank
    .eq("* NO MATCH *")
    .sum()
)


if upc_no_match_count != 14:

    raise ValueError(
        "SMDUPCA '* NO MATCH *' count differs."
    )


numeric_upc = (
    upc_nonblank[
        upc_nonblank
        !=
        "* NO MATCH *"
    ]
)


if len(
    numeric_upc
) != 803:

    raise ValueError(
        "Unexpected numeric UPC count."
    )


if not (
    numeric_upc
    .str.fullmatch(
        r"\d{12}"
    )
    .all()
):

    raise ValueError(
        "Unexpected non-12-digit numeric UPC."
    )


upc_leading_zero_count = int(
    numeric_upc
    .str.startswith("0")
    .sum()
)


if upc_leading_zero_count != 738:

    raise ValueError(
        "SMDUPCA leading-zero count differs."
    )


print(
    "\nPASS: SMDUPCA text fidelity pattern verified."
)

print(
    "      Populated = 817"
)

print(
    "      12-digit numeric UPC = 803"
)

print(
    "      '* NO MATCH *' = 14"
)

print(
    "      Numeric UPC beginning with 0 = 738"
)


# ============================================================
# 20. TEXT FIDELITY - SMD100BR
# ============================================================

brand = (
    df["SMD100BR"]
    .astype(str)
)


brand_nonblank = (
    brand[
        brand != ""
    ]
)


if len(
    brand_nonblank
) != 1256:

    raise ValueError(
        "SMD100BR populated count differs."
    )


# Source has no unwanted leading/trailing whitespace.
if not (
    brand_nonblank
    .str.strip()
    .eq(
        brand_nonblank
    )
    .all()
):

    raise ValueError(
        "Unexpected source brand whitespace."
    )


print(
    "PASS: SMD100BR populated count = 1,256."
)


# ============================================================
# 21. CDC ANCHOR CHECK - SMQ020
# ============================================================

EXPECTED_SMQ020 = {
    1: 2802,
    2: 3127,
    7: 2,
    9: 4
}


actual = (
    df["SMQ020"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_SMQ020:

    raise ValueError(
        "SMQ020 frequency mismatch."
    )


if int(
    df["SMQ020"]
    .isna()
    .sum()
) != 1210:

    raise ValueError(
        "SMQ020 missing count differs."
    )


print(
    "\nPASS: SMQ020 matches CDC."
)


# ============================================================
# 22. CDC ANCHOR CHECK - SMAQUEX2
# ============================================================

EXPECTED_SMAQUEX2 = {
    1: 5935,
    2: 1210
}


actual = (
    df["SMAQUEX2"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_SMAQUEX2:

    raise ValueError(
        "SMAQUEX2 frequency mismatch."
    )


if int(
    df["SMAQUEX2"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SMAQUEX2 missing count differs."
    )


print(
    "PASS: SMAQUEX2 distribution verified."
)


# ============================================================
# 23. BUILD EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
# ONLY:
#
# SMD030   tiny -> 0
# SMD641   tiny -> 0
# SMD100FL tiny -> 0
# SMD100MN tiny -> 0
#
# EVERYTHING ELSE remains source-decoded AS IS.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 24. CLEAN INTEGER-STYLE FIELDS ONLY
# ============================================================

#
# IMPORTANT:
#
# DO NOT convert:
#
# SMD100TR
# SMD100NI
# SMD100CO
# SMDUPCA
# SMD100BR
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Integer/code/count fields cleaned."
)

print(
    "PASS: Measurement fields left as floats."
)

print(
    "PASS: UPC/brand fields left as source strings."
)


# ============================================================
# 25. PRE-EXPORT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in INTEGER_COLUMNS:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )

    else:

        current_values = (
            df[col]
            .to_numpy()
        )


    equal = np.isclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not bool(
        np.all(equal)
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected pre-export numeric changes:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric values equal "
    "expected XPT exactly."
)


# ============================================================
# 26. PRE-EXPORT TEXT FIDELITY
# ============================================================

for col in TEXT_COLUMNS:

    expected_text = (
        expected_df[col]
        .astype(str)
        .tolist()
    )


    current_text = (
        df[col]
        .astype(str)
        .tolist()
    )


    if expected_text != current_text:

        raise ValueError(
            f"{col} changed before export."
        )


print(
    "PASS: Pre-export text fields unchanged."
)


# ============================================================
# 27. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
# round()
# float_format
# UPC numeric conversion
# brand cleanup/trimming
# measurement recalculation
#
# Missing numeric values -> blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 28. RE-READ CSV FOR NUMERIC ROUND-TRIP CHECK
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip",
    dtype={
        "SMDUPCA": "string",
        "SMD100BR": "string"
    }
)


# ============================================================
# 29. STRUCTURE CHECK AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure matches XPT."
)


# ============================================================
# 30. EXACT NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in NUMERIC_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All numeric fields match "
    "expected XPT values exactly."
)


# ============================================================
# 31. NUMERIC MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in NUMERIC_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Numeric missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Numeric missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Numeric missing positions preserved."
)


# ============================================================
# 32. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 7145:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 33. STRICT RAW-TOKEN FIDELITY
# ============================================================

#
# Integer fields:
#
# str(int(source_value))
#
# Measurement fields:
#
# str(float(source_value))
#
# Text:
#
# exact original source string
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if col in TEXT_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(source_value)
            )


        elif col in MEASUREMENT_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )


        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = row[col]


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:
        print(example)


    raise ValueError(
        "SMQ_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected XPT-derived representation."
)


# ============================================================
# 34. UNWANTED .0 IN INTEGER FIELDS
# ============================================================

integer_dot_zero_count = 0
integer_dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            integer_dot_zero_count += 1


            if len(
                integer_dot_zero_examples
            ) < 20:

                integer_dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Unwanted '.0' in integer fields:",
    integer_dot_zero_count
)


if integer_dot_zero_count != 0:

    print(
        integer_dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 remains in integer fields."
    )


print(
    "PASS: No unwanted .0 in "
    "identifier/code/count fields."
)


# ============================================================
# 35. LEGITIMATE MEASUREMENT .0 CHECK
# ============================================================

for col, expected_count in (
    EXPECTED_MEASUREMENT_DOT_ZERO.items()
):

    actual_count = sum(
        row[col].endswith(".0")
        for row in raw_rows
        if row[col] != ""
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: legitimate measurement .0 "
            "formatting changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: Legitimate measurement .0 "
    "values preserved exactly."
)


# ============================================================
# 36. UPC RAW FIDELITY CHECK
# ============================================================

raw_upcs = [
    row["SMDUPCA"]
    for row in raw_rows
    if row["SMDUPCA"] != ""
]


if len(
    raw_upcs
) != 817:

    raise ValueError(
        "Final SMDUPCA populated count changed."
    )


raw_numeric_upcs = [
    value
    for value in raw_upcs
    if value != "* NO MATCH *"
]


if len(
    raw_numeric_upcs
) != 803:

    raise ValueError(
        "Final numeric UPC count changed."
    )


if any(
    len(value) != 12
    for value in raw_upcs
):

    raise ValueError(
        "SMDUPCA length changed."
    )


final_upc_leading_zero_count = sum(
    value.startswith("0")
    for value in raw_numeric_upcs
)


if final_upc_leading_zero_count != 738:

    raise ValueError(
        "Leading-zero UPC values were changed."
    )


final_no_match_count = sum(
    value == "* NO MATCH *"
    for value in raw_upcs
)


if final_no_match_count != 14:

    raise ValueError(
        "SMDUPCA '* NO MATCH *' values changed."
    )


print(
    "PASS: SMDUPCA exact formatting preserved."
)

print(
    "      Leading-zero UPCs preserved = 738"
)

print(
    "      '* NO MATCH *' preserved = 14"
)


# ============================================================
# 37. BRAND RAW FIDELITY CHECK
# ============================================================

raw_brands = [
    row["SMD100BR"]
    for row in raw_rows
    if row["SMD100BR"] != ""
]


if len(
    raw_brands
) != 1256:

    raise ValueError(
        "SMD100BR populated count changed."
    )


print(
    "PASS: SMD100BR text preserved exactly."
)


# ============================================================
# 38. SCIENTIFIC NOTATION CHECK
# ============================================================

#
# Check numeric fields only so text/brand names
# cannot trigger a false positive.
#

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in NUMERIC_COLUMNS:

        value = row[col]

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation "
        "found in numeric fields."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 39. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains."
)


# ============================================================
# 40. FINAL ZERO RECHECK
# ============================================================

for col, expected_count in (
    EXPECTED_ZERO_COUNTS.items()
):

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"Final {col} zero count changed."
        )


print(
    "PASS: Final restored-zero counts exact."
)


# ============================================================
# 41. FINAL MEASUREMENT RECHECK
# ============================================================

for col, cfg in (
    MEASUREMENT_EXPECTATIONS.items()
):

    s = csv_df[col]


    if int(
        s.notna()
        .sum()
    ) != cfg["valid"]:

        raise ValueError(
            f"Final {col} valid count changed."
        )


    if int(
        s.isna()
        .sum()
    ) != cfg["missing"]:

        raise ValueError(
            f"Final {col} missing count changed."
        )


    if not np.isclose(
        s.min(),
        cfg["min"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"Final {col} minimum changed."
        )


    if not np.isclose(
        s.max(),
        cfg["max"],
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"Final {col} maximum changed."
        )


print(
    "PASS: Final cigarette-content "
    "measurements unchanged."
)


# ============================================================
# 42. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("SMQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "SMD030   = 122"
)

print(
    "SMD641   = 123"
)

print(
    "SMD100FL = 7"
)

print(
    "SMD100MN = 882"
)

print(
    "TOTAL    = 1,134"
)


print(
    "\nMeasurement fields preserved AS IS:"
)

print(
    "SMD100TR"
)

print(
    "SMD100NI"
)

print(
    "SMD100CO"
)


print(
    "\nSMD100NI genuine fractional values = 872"
)

print(
    "SMD100NI legitimate '.0' values = 62"
)


print(
    "\nText fields preserved exactly:"
)

print(
    "SMDUPCA"
)

print(
    "SMD100BR"
)

print(
    "SMDUPCA leading-zero UPCs = 738"
)

print(
    "SMDUPCA '* NO MATCH *' = 14"
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Numeric missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0 in integer fields:",
    integer_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "SMQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 SMQ_E XPT -> CSV
Rows: 7,145
Columns: 37
PASS: SMQ_E structure = 7,145 rows x 37 columns.

Integer/code/count fields: 32
Measurement fields preserved AS IS: ['SMD100TR', 'SMD100NI', 'SMD100CO']
Text fields preserved AS IS: ['SMDUPCA', 'SMD100BR']

--- ORIGINAL XPT ZERO CHECK ---
True decoded numeric zeros: 0
Tiny XPORT artifacts: 1,134

Tiny values by variable:
SMD100MN    882
SMD641      123
SMD030      122
SMD100FL      7

PASS: Exact tiny-value pattern confirmed.
PASS: All tiny values occur in CDC zero-permitted variables.

--- VALIDATED ZERO RESTORATION ---
SMD030    : 122 tiny -> 0
SMD641    : 123 tiny -> 0
SMD100FL  : 7 tiny -> 0
SMD100MN  : 882 tiny -> 0
PASS: All 1,134 validated zeros restored.
PASS: Final zero counts exact.
PASS: No unexpected zeros introduced.

Remaining tiny artifacts: 0
PASS: No tiny artifacts remain.

PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL VALUE CHECK ---
{'SMD100NI': 872}
PASS: SM